# 基礎設定

## SQLite

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db') # 如果資料庫不存在，會自動幫你建立

# 建立 stock_date 資料表
sql_create_table_stock_date = """
CREATE TABLE IF NOT EXISTS `stock_date` (
	`stock_id` TEXT,
	`date`	TEXT,
	`open`	REAL,
	`high`	REAL,
	`low`	REAL,
	`close`	REAL,
	`volume` INTEGER
)
"""
cursor = conn.execute(sql_create_table_stock_date)

# 建立 stock_list 資料表
#     多一個自動編號欄位 `idx` 為 PRIMARY KEY
#     AUTOINCREMENT：自動編號，每新增一筆就 +1，永不重複
sql_create_table_stock_list = """
CREATE TABLE IF NOT EXISTS `stock_list` (
	`idx`	INTEGER PRIMARY KEY AUTOINCREMENT,
	`stock_id`	TEXT,
	`name`	TEXT
)
"""
cursor = conn.execute(sql_create_table_stock_list)

sql_create_ui = """
    CREATE UNIQUE INDEX IF NOT EXISTS `id_date` ON `stock_date`(`stock_id`, `date`)
"""
conn.execute(sql_create_ui)

conn.close()

In [ ]:
import sqlite3

def insert_safe(sql_ins):
    conn = sqlite3.connect('stocks.db')
    try:
        conn.execute(sql_ins)
        conn.commit()
        print("新增成功")
    except sqlite3.IntegrityError as e:
        print(f"違反資料庫目前的限制，不新增此筆資料")
    conn.close()

stock_infos = [
    ['NVDA' ,'NVIDIA'],
    ['AAPL' ,'APPLE'],
    ['GOOGL', 'GOOGLE']
]

for stock_id, stock_name in stock_infos:
    sql_ins = f"""
        INSERT INTO `stock_list` (`stock_id`, `name`)
        VALUES ( '{stock_id}' ,'{stock_name}' )
    """
    insert_safe(sql_ins)

In [ ]:
import sqlite3
def insert_safe(sql_ins):
    conn = sqlite3.connect('stocks.db')
    try:
        conn.execute(sql_ins)
        conn.commit()
        print("新增成功")
    except sqlite3.IntegrityError as e:
        print(f"違反資料庫目前的限制，不新增此筆資料")
    conn.close()

datas_NVDA = [
    ['2026-04-20', '199.98', '202.17', '197.84', '202.06', '119381400'],
    ['2026-04-21', '202.13', '202.75', '199', '199.88', '107945302'],
    ['2026-04-22', '200.99', '202.5', '199', '202.5', '107501042'],
    ['2026-04-23', '202.46', '203.83', '197.22', '199.64', '113561830'],
    ['2026-04-24', '199.96', '210.965', '199.81', '208.26', '166563954']
]
datas_AAPL = [
    ['2026-04-20', '270.33', '274.27', '270.29', '273.05', '36590200'],
    ['2026-04-21', '271.5', '272.8', '265.4', '266.17', '50209800'],
    ['2026-04-22', '267.82', '273.74', '266.87', '273.17', '43249204'],
    ['2026-04-23', '275.05', '275.77', '271.65', '273.43', '33399639'],
    ['2026-04-24', '272.76', '273.06', '269.67', '270.11', '20541211']
]
def ins_stock_data(stock_id, datas):
    for row in datas:
        d, o, h, l, c, v = row
        sql_ins = """
            INSERT INTO `stock_date` (`stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`)
            VALUES ( '{}' ,'{}', {}, {}, {}, {}, {} )
        """.format(stock_id, d, float(o), float(h), float(l), float(c), int(v))
        insert_safe(sql_ins)

ins_stock_data('NVDA', datas_NVDA)
ins_stock_data('AAPL', datas_AAPL)

## Flask

In [ ]:
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN"

In [ ]:
!pip install pyngrok flask --quiet

In [ ]:
home_html = "Welcome to Stock Board <br> <a href='/stock_list'>進入股票列表頁</a>"

stock_list_html = """
<a href='/'>回首頁</a>
<table>
    <thead>
        <tr>
            <th>股票代碼</th>
            <th>股票名</th>
            <th>股票頁</th>
        </tr>
    </thead>
    <tbody>
        {% for row in datas %}
        <tr>
            <td>{{ row[0] }}</td>
            <td>{{ row[1] }}</td>
            <td><a href="/stock">進入股票頁</a></td>
        </tr>
        {% endfor %}
    </tbody>
</table>
"""

stock_html = """
<h1>股票代碼: {{ stock_id }}</h1>
<a href='/stock_list'>回股票列表頁</a>
<table>
    <thead>
        <tr>
            <th>日期</th>
            <th>開盤價</th>
            <th>最高價</th>
            <th>最低價</th>
            <th>收盤價</th>
            <th>成交量</th>
        </tr>
    </thead>
    <tbody>
        {% for row in datas %}
        <tr>
            <td>{{ row[0] }}</td>
            <td>{{ row[1] }}</td>
            <td>{{ row[2] }}</td>
            <td>{{ row[3] }}</td>
            <td>{{ row[4] }}</td>
            <td>{{ row[5] }}</td>
        </tr>
        {% endfor %}
    </tbody>
</table>
"""

In [ ]:
import os

dir_name = "templates"
if dir_name not in os.listdir():
    os.makedirs(dir_name)

with open("templates/home.html","w") as fo:
    fo.write(home_html)

with open("templates/stock_list.html","w") as fo:
    fo.write(stock_list_html)

with open("templates/stock.html","w") as fo:
    fo.write(stock_html)

# SQLite

## CRUD

### Create 新增資料

#### [練習] 新增 NVDA 的每日股票資料
請多增加幾筆 NVDA 的股票資訊 到 `stock_list`

In [ ]:
import sqlite3

def insert_safe(sql_ins):
    conn = sqlite3.connect('stocks.db')
    try:
        conn.execute(sql_ins)
        conn.commit()
        print("新增成功")
    except sqlite3.IntegrityError as e:
        print(f"違反資料庫目前的限制，不新增此筆資料")
    conn.close()

def ins_stock_data(stock_id, datas):
    for row in datas:
        d, o, h, l, c, v = row
        sql_ins = """
            INSERT INTO `stock_date` (`stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`)
            VALUES ( '{}' ,'{}', {}, {}, {}, {}, {} )
        """.format(stock_id, d, float(o), float(h), float(l), float(c), int(v))
        insert_safe(sql_ins)

insert_datas_NVDA = [
    ['2026-04-16', '197.43', '199.85', '195.81', '198.35', '134012900'],
    ['2026-04-17', '199.9', '201.7', '199.27', '201.68', '160324417']
]

ins_stock_data("NVDA", insert_datas_NVDA)

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = "SELECT * FROM `stock_date`"
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

### Read 讀取資料（進階：篩選與排序）

#### [練習] NVIDIA 股價資料整理與最高價篩選
- 列出 NVIDIA (NVDA) 2026-04-21 ~ 2026-04-24 的每日資訊
    - 日期
    - 開盤、收盤、最高、最低
    - **均價 CDP**：`(最高 + 最低 + 2 * 收盤) / 4`
- 列出擁有最大「最高價」的兩筆資料

> 提示：`ORDER BY` 搭配 `LIMIT` 可以快速取出最大的幾筆。

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT `date`, `open`, `close`, `high`, `low`, (`high`+`low`+2*`close`)/4
    FROM `stock_date`
    WHERE `stock_id` = 'NVDA' and `date` >= '2026-04-21' and `date` <= '2026-04-24'
    ORDER BY `date`
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

列出擁有最大「最高價」的兩筆資料

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT `stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`
    FROM `stock_date`
    ORDER BY `high` DESC
    LIMIT 2
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

# Flask

## 串接 SQLite 資料庫

### [練習] 增加 `date_start` 參數
- 增加一個參數 `date_start`
    - 如果走 `get_stock_path`，url 設計成 `/stock/NVDA/YYYY-MM-DD`
    - 如果走 `get_stock_param`，url 設計成 `/stock?stock_id=NVDA&date_start=YYYY-MM-DD`
- 先把收到的 `date_start` `print` 出來確認有正確接到
- 把 `date_start` 當作 SQL 查詢條件：搜尋此日期（包含）之後的每日資訊

#### 提示：  
- path 版的網頁路徑和函式傳入的參數要增加 `date_start`   
- query 版的 `date_start` 則是要用 request.args.get 取得，是在函式內增加一行程式碼。
- 在 SQLite 是以 TEXT 儲存，WHERE 條件記得用單引號：`` `date` >= 'YYYY-MM-DD' ``
  
  
> 想想看：如果 `date_start` 沒傳入（為 `None`）時，SQL 該怎麼組才不會出錯？

In [ ]:
import os

dir_name = "templates"
if dir_name not in os.listdir():
    os.makedirs(dir_name)

home_with_date_html = "Welcome to Stock Board <br> <a href='/stock_list?date_start=2026-04-22'>進入股票列表頁</a>"

stock_list_with_date_html = """
<a href='/'>回首頁</a>
<table>
    <thead>
        <tr>
            <th>股票代碼</th>
            <th>股票名</th>
            <th>股票頁(路徑)</th>
            <th>股票頁(參數)</th>
        </tr>
    </thead>
    <tbody>
        {% for row in datas %}
        <tr>
            <td>{{ row[0] }}</td>
            <td>{{ row[1] }}</td>
            <td><a href="/stock/{{ row[0] }}/{{ date_start }}">進入股票頁(路徑)</a></td>
            <td><a href="/stock?stock_id={{ row[0] }}&date_start={{ date_start }}">進入股票頁(參數)</a></td>
        </tr>
        {% endfor %}
    </tbody>
</table>
"""

with open("templates/home_with_date.html","w") as fo:
    fo.write(home_with_date_html)

with open("templates/stock_list_with_date.html","w") as fo:
    fo.write(stock_list_with_date_html)

In [ ]:
from flask import Flask, render_template, request
from pyngrok import ngrok
import sqlite3
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

def get_stock_datas(stock_id, date_start):
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `date`, `open`, `high`, `low`, `close`, `volume`
        FROM `stock_date`
        WHERE `stock_id` = '{}' and `date` >= '{}'
        ORDER BY `date`
    """.format(stock_id, date_start)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    conn.close()
    return datas

def get_stock_list():
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `stock_id`, `name`
        FROM `stock_list`
        WHERE `stock_id` in ('NVDA', 'AAPL')
    """.format(stock_id)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    return datas

@app.route("/")
def home():
    return render_template("home_with_date.html")

@app.route("/stock_list")
def stock_list():
    date_start = request.args.get('date_start')
    return render_template("stock_list_with_date.html", date_start=date_start, datas = get_stock_list())

@app.route("/stock/<stock_id>")
def get_stock_path(stock_id):
    return render_template("stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

@app.route("/stock/<stock_id>/<date_start>")
def get_stock_date_path(stock_id, date_start):
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    date_start = request.args.get('date_start')
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

In [ ]:
from flask import Flask, render_template, request
from pyngrok import ngrok
import sqlite3
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

def get_stock_datas(stock_id, date_start):
    conn = sqlite3.connect('stocks.db')
    print(type(date_start), date_start)
    if date_start=="None":
        sql = """
            SELECT `date`, `open`, `high`, `low`, `close`, `volume`
            FROM `stock_date`
            WHERE `stock_id` = '{}'
            ORDER BY `date`
        """.format(stock_id)
    else:
        sql = """
            SELECT `date`, `open`, `high`, `low`, `close`, `volume`
            FROM `stock_date`
            WHERE `stock_id` = '{}' and `date` >= '{}'
            ORDER BY `date`
        """.format(stock_id, date_start)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    conn.close()
    return datas

def get_stock_list():
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `stock_id`, `name`
        FROM `stock_list`
        WHERE `stock_id` in ('NVDA', 'AAPL')
    """.format(stock_id)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    return datas

@app.route("/")
def home():
    return render_template("home_with_date.html")

@app.route("/stock_list")
def stock_list():
    date_start = request.args.get('date_start')
    return render_template("stock_list_with_date.html", date_start=date_start, datas = get_stock_list())

@app.route("/stock/<stock_id>")
def get_stock_path(stock_id):
    return render_template("stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

@app.route("/stock/<stock_id>/<date_start>")
def get_stock_date_path(stock_id, date_start):
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    date_start = request.args.get('date_start')
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()